# Download OpenAI vector store files

This notebook reads OpenAI credentials from `bhtom3/.bhtom.env`, discovers the vector stores attached to the configured assistant, and downloads all files to a local folder.


In [5]:
from pathlib import Path
import json
from dotenv import dotenv_values

REPO_ROOT = Path("/Users/wyrzykow/Documents/GitHub/bhtom2")
ENV_PATH = Path("/Users/wyrzykow/Documents/GitHub/bhtom3/.bhtom.env")

if not REPO_ROOT.exists():
    raise RuntimeError(f"Missing repo root: {REPO_ROOT}")

if not ENV_PATH.exists():
    raise RuntimeError(f"Missing env file: {ENV_PATH}")

secret = dotenv_values(ENV_PATH)
OPENAI_API_KEY = secret.get("OPENAI_API_KEY", "")
BHTOM_ASSISTANT_ID = secret.get("BHTOM_ASSISTANT_ID", "")

print("Repo root:", REPO_ROOT)
print("Env path:", ENV_PATH)
print("Assistant ID:", BHTOM_ASSISTANT_ID)
print("API key loaded:", bool(OPENAI_API_KEY))


Repo root: /Users/wyrzykow/Documents/GitHub/bhtom2
Env path: /Users/wyrzykow/Documents/GitHub/bhtom3/.bhtom.env
Assistant ID: asst_zf053tMtsIWA4Jh5dsR339QP
API key loaded: True


In [6]:
import requests
from openai import OpenAI

if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY is not configured in bhtom3/.bhtom.env.")

if not BHTOM_ASSISTANT_ID:
    raise RuntimeError("BHTOM_ASSISTANT_ID is not configured in bhtom3/.bhtom.env.")

client = OpenAI(api_key=OPENAI_API_KEY)
headers = {"Authorization": f"Bearer {OPENAI_API_KEY}"}


In [7]:
assistant = client.beta.assistants.retrieve(BHTOM_ASSISTANT_ID)
tool_resources = getattr(assistant, "tool_resources", None)
file_search = getattr(tool_resources, "file_search", None) if tool_resources else None
vector_store_ids = list(getattr(file_search, "vector_store_ids", []) or [])

if not vector_store_ids:
    raise RuntimeError("No vector stores attached to this assistant.")

print("Assistant:", assistant.id)
print("Vector stores:", vector_store_ids)


/var/folders/zg/pgrq7x2x5ddbcvb08s5m2nf80000gq/T/ipykernel_12957/4286028470.py:1: DeprecationWarning: deprecated
  assistant = client.beta.assistants.retrieve(BHTOM_ASSISTANT_ID)


Assistant: asst_zf053tMtsIWA4Jh5dsR339QP
Vector stores: ['vs_68ca65d536c0819191ab8ac94f45b9c6']


In [8]:
download_root = REPO_ROOT / "notebooks" / "downloaded_vector_store_files"
download_root.mkdir(parents=True, exist_ok=True)

manifest = []

for vector_store_id in vector_store_ids:
    store_dir = download_root / vector_store_id
    store_dir.mkdir(parents=True, exist_ok=True)

    files_page = client.vector_stores.files.list(vector_store_id=vector_store_id, limit=100)
    files = list(files_page.data)

    print(f"Vector store {vector_store_id}: {len(files)} file(s)")

    for vs_file in files:
        vector_store_file_id = getattr(vs_file, "id", None)
        file_id = getattr(vs_file, "file_id", None) or vector_store_file_id

        file_obj = client.files.retrieve(file_id)
        filename = getattr(file_obj, "filename", None) or f"{file_id}.bin"
        target_path = store_dir / filename

        original_download_error = None
        response = requests.get(
            f"https://api.openai.com/v1/files/{file_id}/content",
            headers=headers,
            timeout=120,
        )
        if response.ok:
            target_path.write_bytes(response.content)
            local_path = str(target_path.relative_to(REPO_ROOT))
        else:
            local_path = None
            original_download_error = f"{response.status_code} {response.text}"

        parsed_text_path = store_dir / f"{Path(filename).stem}__parsed.txt"
        parsed_response = requests.get(
            f"https://api.openai.com/v1/vector_stores/{vector_store_id}/files/{file_id}/content",
            headers=headers,
            timeout=120,
        )

        parsed_preview = None
        parsed_download_error = None
        if parsed_response.ok:
            parsed_text_path.write_text(parsed_response.text, encoding="utf-8")
            parsed_preview = str(parsed_text_path.relative_to(REPO_ROOT))
        else:
            parsed_download_error = f"{parsed_response.status_code} {parsed_response.text}"

        manifest.append(
            {
                "vector_store_id": vector_store_id,
                "vector_store_file_id": vector_store_file_id,
                "file_id": file_id,
                "filename": filename,
                "local_path": local_path,
                "parsed_text_path": parsed_preview,
                "original_download_error": original_download_error,
                "parsed_download_error": parsed_download_error,
            }
        )

        if local_path:
            print(f"  downloaded original: {filename}")
        else:
            print(f"  original download unavailable for {filename}")

        if parsed_preview:
            print(f"  downloaded parsed text: {parsed_text_path.name}")
        else:
            print(f"  parsed text download unavailable for {filename}")

manifest_path = download_root / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Manifest written to", manifest_path)


Vector store vs_68ca65d536c0819191ab8ac94f45b9c6: 5 file(s)
  original download unavailable for DocumentationAPI.md
  downloaded parsed text: DocumentationAPI__parsed.txt
  original download unavailable for TargetDetailPage.md
  downloaded parsed text: TargetDetailPage__parsed.txt
  original download unavailable for FAQ.md
  downloaded parsed text: FAQ__parsed.txt
  original download unavailable for CreateTarget.md
  downloaded parsed text: CreateTarget__parsed.txt
  original download unavailable for DataProductUploadForm.md
  downloaded parsed text: DataProductUploadForm__parsed.txt
Manifest written to /Users/wyrzykow/Documents/GitHub/bhtom2/notebooks/downloaded_vector_store_files/manifest.json


In [9]:
manifest


[{'vector_store_id': 'vs_68ca65d536c0819191ab8ac94f45b9c6',
  'vector_store_file_id': 'file-G2KykuY7EwbN8njHtmzFoV',
  'file_id': 'file-G2KykuY7EwbN8njHtmzFoV',
  'filename': 'DocumentationAPI.md',
  'local_path': None,
  'parsed_text_path': 'notebooks/downloaded_vector_store_files/vs_68ca65d536c0819191ab8ac94f45b9c6/DocumentationAPI__parsed.txt',
  'original_download_error': '400 {\n  "error": {\n    "message": "Not allowed to download files of purpose: assistants",\n    "type": "invalid_request_error",\n    "param": null,\n    "code": null\n  },\n  "detail": {\n    "message": "Not allowed to download files of purpose: assistants",\n    "code": null\n  }\n}\n',
  'parsed_download_error': None},
 {'vector_store_id': 'vs_68ca65d536c0819191ab8ac94f45b9c6',
  'vector_store_file_id': 'file-LVPxuSu2QqgRu5TEGFnphp',
  'file_id': 'file-LVPxuSu2QqgRu5TEGFnphp',
  'filename': 'TargetDetailPage.md',
  'local_path': None,
  'parsed_text_path': 'notebooks/downloaded_vector_store_files/vs_68ca65d53